In [0]:
from pyspark.sql.functions import *

In [0]:
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
   "<YOUR KEY>"
)


In [0]:
fact_sales_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/facts/fact_sales"
dim_prods_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/dimensions/dim_products"

In [0]:
fs_df = spark.read.format("delta").load(fact_sales_path)

dim_prod_df = spark.read.format("delta").load(dim_prods_path)

In [0]:
fs_df.printSchema()

dim_prod_df.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- date_key: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- sale_key: long (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- product_key: long (nullable = true)



In [0]:
product_sales_base = fs_df.alias("f").join(dim_prod_df.alias('c'), on="product_key", how="left").withColumn(
        "product_category_name",
        when(
            col("product_category_name").isNull(),
            "unknown_category"
        ).otherwise(
            col("product_category_name")
        )
    )

In [0]:
prod_sales = product_sales_base.groupBy("product_category_name").agg(
    countDistinct("order_id").alias("total_orders"),
    sum("total_installments").alias("total_revenue")
)

In [0]:
display(prod_sales)

product_category_name,total_orders,total_revenue
unknown_category,1451,4304
pcs,181,1322
bebes,2885,9227
artes,202,490
cine_foto,65,189
moveis_decoracao,6449,27287
pc_gamer,8,25
construcao_ferramentas_construcao,748,3478
tablets_impressao_imagem,79,287
fashion_roupa_masculina,112,384


In [0]:
from common_scripts.dq_framework import *

run_dq_check(
    prod_sales,
    "product_revenue",
    "null_total_revenue",
    col("total_revenue").isNull(),
    severity="critical"
)

run_dq_check(
    prod_sales,
    "product_revenue",
    "null_total_orders",
    col("total_orders").isNull(),
    severity="critical"
)


[CRITICAL] product_revenue | null_total_revenue: 0
[CRITICAL] product_revenue | null_total_orders: 0


0

In [0]:
dq_df = dq_df_from_dq_results(spark, dq_results)

dq_path = f"abfss://audit@{storage_account}.dfs.core.windows.net/gold_dq_logs/kpi_product_revenue"

dq_df.write.format('delta').mode('append').save(dq_path)

In [0]:
prod_revenue_path =  f"abfss://gold@{storage_account}.dfs.core.windows.net/kpi/kpi_product_revenue"


prod_sales.write.format('delta').mode('overwrite').save(prod_revenue_path)

In [0]:
pr = spark.read.format('delta').load(prod_revenue_path)

display(pr)

product_category_name,total_orders,total_revenue
unknown_category,1451,4304
pcs,181,1322
bebes,2885,9227
artes,202,490
cine_foto,65,189
moveis_decoracao,6449,27287
pc_gamer,8,25
construcao_ferramentas_construcao,748,3478
tablets_impressao_imagem,79,287
fashion_roupa_masculina,112,384
